In [33]:
import numpy as np
import pandas as pd

#Settings
CSV_PATH = "bank-additional-full.csv"   # change if needed
SEPARATOR = ";" # dataset uses semicolons
LEARNING_RATE = 0.3
EPOCHS = 2000
L2_REG = 0.0        # set >0 for L2 regularization
PRINT_EVERY = 200   # how often to print loss during training
RANDOM_SEED = 42


# 1. Load data
df = pd.read_csv(CSV_PATH, sep=SEPARATOR)
print("Loaded dataframe shape:", df.shape)
print("Columns:", df.columns.tolist())


# 2. Target encoding
# Convert 'y' from 'yes'/'no' to 1/0
if df['y'].dtype == object:
    df['y'] = df['y'].map({'yes': 1, 'no': 0})
df['y'] = df['y'].astype(int)


# 3. Identify categorical and numeric features
categorical_cols = [c for c in df.select_dtypes(include=['object']).columns.tolist() if c != 'y']
numeric_cols = [c for c in df.columns.tolist() if c not in categorical_cols + ['y']]

print("Categorical cols:", categorical_cols)
print("Numeric cols:", numeric_cols)

# 4. One-hot encode categorical features
if len(categorical_cols) > 0:
    df_cat = pd.get_dummies(df[categorical_cols], drop_first=False)  # keep all dummies
else:
    df_cat = pd.DataFrame(index=df.index)

# 5. Standardize numeric features
df_num = df[numeric_cols].copy()
# Convert to numeric if some numeric columns are read as object due to stray values
for col in df_num.columns:
    df_num[col] = pd.to_numeric(df_num[col], errors='coerce')


if df_num.isnull().any().any():
    df_num = df_num.fillna(df_num.mean())


num_means = df_num.mean()
num_stds = df_num.std(ddof=0).replace(0, 1.0)  # avoid zero-division
df_num_scaled = (df_num - num_means) / num_stds


# 6. Prepare feature matrix X and target vector y
X_df = pd.concat([df_num_scaled, df_cat], axis=1)


X_df.insert(0, "intercept", 1.0)


X = X_df.values.astype(float)      
y = df['y'].values.reshape(-1, 1).astype(float)  

N, D = X.shape
print(f"Prepared X shape: {X.shape}, y shape: {y.shape}")


# 7. Numeric-stable sigmoid and loss
def sigmoid(z):
    # avoid overflow when exponentiating
    z = np.clip(z, -250, 250)
    return 1.0 / (1.0 + np.exp(-z))

def log_loss(y_true, y_prob, eps=1e-12):
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))


# 8. Training with gradient descent
def train(X, y, lr=0.1, epochs=1000, l2_reg=0.0, verbose=True):
    np.random.seed(RANDOM_SEED)
    N, D = X.shape
    w = np.zeros((D, 1), dtype=float)

    reg_mask = np.ones((D, 1), dtype=float)
    reg_mask[0, 0] = 0.0

    for epoch in range(1, epochs + 1):
        # predictions
        logits = X @ w   # shape (N,1)
        y_pred = sigmoid(logits)

        # gradient (D x 1)
        grad = (1.0 / N) * (X.T @ (y_pred - y))

        if l2_reg > 0:
            grad += (l2_reg / N) * (w * reg_mask)

        # gradient descent step
        w -= lr * grad

        if verbose and (epoch % PRINT_EVERY == 0 or epoch == 1 or epoch == epochs):
            loss = log_loss(y, y_pred)
            reg_term = 0.5 * (l2_reg / N) * np.sum((w * reg_mask) ** 2) if l2_reg > 0 else 0.0
            print(f"Epoch {epoch}/{epochs} - loss: {loss:.6f} - reg_term: {reg_term:.6f}")

    return w


# 9. Predict & evaluate helpers
def predict_proba(X, w):
    return sigmoid(X @ w)

def predict(X, w, threshold=0.5):
    return (predict_proba(X, w) >= threshold).astype(int)

def confusion_matrix(y_true, y_pred):
    y_true = y_true.flatten().astype(int)
    y_pred = y_pred.flatten().astype(int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    return {"TP": tp, "TN": tn, "FP": fp, "FN": fn}

# 10. Train model
w = train(X, y, lr=LEARNING_RATE, epochs=EPOCHS, l2_reg=L2_REG, verbose=True)


# 11. Evaluate
y_proba = predict_proba(X, w)
y_pred = (y_proba >= 0.5).astype(int)

acc = (y_pred == y).mean()
cm = confusion_matrix(y, y_pred)

print("\nFinal evaluation on training data:")
print(f"Accuracy: {acc:.4f}")
print("Confusion matrix (train):", cm)

# 12. Inspect a few coefficients
coef_series = pd.Series(w.flatten(), index=X_df.columns)
print("\nTop 10 absolute-weight features:")
print(coef_series.abs().sort_values(ascending=False).head(10))

Loaded dataframe shape: (41188, 21)
Columns: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']
Categorical cols: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
Numeric cols: ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Prepared X shape: (41188, 64), y shape: (41188, 1)
Epoch 1/2000 - loss: 0.693147 - reg_term: 0.000000
Epoch 200/2000 - loss: 0.212799 - reg_term: 0.000000
Epoch 400/2000 - loss: 0.210892 - reg_term: 0.000000
Epoch 600/2000 - loss: 0.210120 - reg_term: 0.000000
Epoch 800/2000 - loss: 0.209668 - reg_term: 0.000000
Epoch 1000/2000 - loss: 0.209366 - reg_term: 0.000000
Epoch 1200/2000 - loss: 0.209149 - reg_term: 0.000000
Epoch 1400/2000